In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import operator
import random
import time
import json
from datetime import datetime
import pandas as pd
import config
from langchain_aws import ChatBedrock
import boto3

# ── Disposition taxonomy ───────────────────────────────────────────────────────
df = pd.read_excel("Disposition_L1_L2.xlsx")
DISPOSITION_TAXONOMY = df[["Level 1", "Level 2", "Description"]].to_string(index=False)

# ── L3 flags with descriptions ─────────────────────────────────────────────────
df_l3 = pd.read_excel("Disposition_L3_Flags.xlsx")
L3_OPTIONS = "\n".join(
    f"- {row['L3 Label']}: {row['Description']}"
    for _, row in df_l3.iterrows()
)

# ── LLMs ──────────────────────────────────────────────────────────────────────
# generator_llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     google_api_key=config.GOOGLE_API_KEY,
#     temperature=1.8,
# )

generator_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=config.ANTHROPIC_API_KEY,
    temperature=0.8,
)
evaluator_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=config.ANTHROPIC_API_KEY,
    temperature=0.2,
)

# evaluator_llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     google_api_key=config.GOOGLE_API_KEY,
#     temperature=0.2,
# )

# bedrock_client = boto3.client(
#     service_name="bedrock-runtime",
#     region_name=config.AWS_REGION,
#     aws_access_key_id=config.AWS_ACCESS_KEY_ID,
#     aws_secret_access_key=config.AWS_SECRET_ACCESS_KEY,
# )

# generator_llm = ChatBedrock(model_id="amazon.nova-pro-v1:0", region_name="us-east-1")
# evaluator_llm = ChatBedrock(model_id="amazon.nova-pro-v1:0", region_name="us-east-1")
# optimizer_llm = ChatBedrock(model_id="amazon.nova-pro-v1:0", region_name="us-east-1")

# ── Rules: Contacted ───────────────────────────────────────────────────────────
CONTACTED_RULES = (
    "- Diarise every turn strictly as 'Agent:' or 'Customer:'\n"
    "- Write entirely in {language}\n"
    "- SCRIPT: Always use Roman script (A-Z letters). Do NOT use Devanagari or any other non-Latin script, even for Hindi or Hinglish.\n"
    "- Internally invent all details — a realistic Indian customer name, loan amount, due date,\n"
    "  and account context — woven naturally into the conversation. Do not list them as headers or metadata.\n"
    "- The conversation must organically lead to the given L1 and L2 as its outcome.\n"
    "- On avg 2500 characters per transcript, max 4500 characters\n"
    "- Include natural noise: interruptions (<interruption>), incomplete sentences, repetition, code-switching mid-sentence\n"
    "- 4-8 turns for engaged calls\n"
)

# ── Rules: Not Contacted ───────────────────────────────────────────────────────
NON_CONTACTED_RULES = (
    "- The agent dials but NEVER speaks. There must be NO 'Agent:' turn anywhere in the transcript.\n"
    "- The entire transcript is a SINGLE 'Customer:' turn containing ONLY a realistic telecom\n"
    "  or voicemail system announcement — exactly what the agent hears on their end.\n"
    "- SCRIPT: Always use Roman script (A-Z letters). Do NOT use Devanagari or any other non-Latin script, even for Hindi or Hinglish.\n"
    "- Write the announcement in {language} using Roman script.\n"
    "- Choose the most realistic announcement for the specific L2 outcome:\n"
    "\n"
    "  No answer / ringing out:\n"
    "    Hindi    -> Customer: App abhi jis vyakti ko call kar rahe hain vo abhi uttar nahi de rahe. Kripya thodi der baad dobara call karein.\n"
    "    Hinglish -> Customer: Jis number pe aap call kar rahe hain, wo abhi available nahi hai. Please thodi der baad try karein.\n"
    "    English  -> Customer: The person you are trying to reach is not answering. Please try again later.\n"
    "\n"
    "  Voicemail / missed call box:\n"
    "    Hindi    -> Customer: Aapka call [naam] ke voicemail mein ja raha hai. Beep ke baad apna sandesh chhod dein.\n"
    "    Hinglish -> Customer: You have reached the voicemail of [naam]. Please leave a message after the beep.\n"
    "    English  -> Customer: You have reached the voicemail of [name]. Please leave a message after the tone.\n"
    "\n"
    "  Number switched off / out of coverage:\n"
    "    Hindi    -> Customer: Aapka call abhi nahi lag sakta. Yeh phone band hai ya coverage area se bahar hai.\n"
    "    Hinglish -> Customer: Is number ka phone switched off hai ya coverage area se bahar hai. Please baad mein try karein.\n"
    "    English  -> Customer: The number you have dialled is currently switched off or out of coverage area.\n"
    "\n"
    "  Number does not exist / wrong number:\n"
    "    Hindi    -> Customer: Aapne jo number dial kiya hai vo maujood nahi hai. Kripya number check karke dobara dial karein.\n"
    "    Hinglish -> Customer: Yeh number exist nahi karta. Please number check karke dobara dial karein.\n"
    "    English  -> Customer: The number you have dialled does not exist. Please check the number and try again.\n"
    "\n"
    "  Call rejected / declined:\n"
    "    Hindi    -> Customer: Aapka call reject kar diya gaya hai.\n"
    "    Hinglish -> Customer: Call reject ho gayi hai.\n"
    "    English  -> Customer: Your call has been declined.\n"
    "\n"
    "- Replace [naam] / [name] with a realistic invented Indian name.\n"
    "- Keep the transcript to exactly 1 line — no extra dialogue, no agent turns whatsoever.\n"
)

# ── Batch config ───────────────────────────────────────────────────────────────
BATCH_SIZE      = 100
MAX_ITERATION   = 2
MAX_CONCURRENCY = 25

# ── Pydantic schemas ───────────────────────────────────────────────────────────
class TranscriptOutput(BaseModel):
    transcript: str = Field(..., description="The full diarised call transcript. Must use Roman script only — no Devanagari.")
    call_stage: str = Field(..., description="Collection stage e.g. Early Bucket, Mid Bucket, NPA, Legal.")
    dialect:    str = Field(..., description="Regional Indian dialect of the customer e.g. Mumbai Hindi, Punjabi, Tamil-accented English.")
    l3:         str = Field(..., description="Pipe-separated L3 labels from the provided options that apply to this interaction.")

class TranscriptEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback:   str = Field(..., description="Feedback for the transcript.")

structured_generator_llm = generator_llm.with_structured_output(TranscriptOutput)
structured_evaluator_llm  = evaluator_llm.with_structured_output(TranscriptEvaluation)

# ── State ──────────────────────────────────────────────────────────────────────
class TranscriptState(TypedDict):
    language:    Literal["Hindi", "Hinglish", "English"]
    l1:          str
    l2:          str
    description: str
    l3:          str
    call_stage:  str
    dialect:     str
    transcript:  str
    evaluation:  Literal["approved", "needs_improvement"]
    feedback:    str
    iteration:   int
    max_iteration: int
    transcript_history: Annotated[list[str], operator.add]
    feedback_history:   Annotated[list[str], operator.add]

# ── Contacted prompt variants ──────────────────────────────────────────────────

def contacted_prompt_v1(state: dict, rules: str) -> str:
    return (
        f"Generate a realistic debt collection call transcript in {state['language']} using Roman script only (no Devanagari).\n\n"
        f"Outcome to reflect:\n"
        f"- L1          : {state['l1']}\n"
        f"- L2          : {state['l2']}\n"
        f"- Description : {state['description']}\n\n"
        f"Use the Description to shape the scenario, tone, and context of the call.\n\n"
        f"Also assign:\n"
        f"- call_stage : the collection stage this call belongs to (e.g. Early Bucket, Mid Bucket, NPA, Legal)\n"
        f"- dialect    : the regional Indian dialect the customer speaks "
        f"(e.g. Mumbai Hindi, Punjabi-accented, Tamil-accented English, Bengali-accented Hindi)\n"
        f"- l3         : select all applicable L3 labels from the list below that are evidenced "
        f"in the interaction (pipe-separated)\n\n"
        f"L3 Options (label: description):\n{L3_OPTIONS}\n\n"
        f"Rules:\n{rules}\n"
    )


def contacted_prompt_v2(state: dict, rules: str) -> str:
    return (
        f"You are building a training dataset for an Indian loan collections AI.\n"
        f"Write one realistic phone call transcript in {state['language']} using Roman script only (no Devanagari), "
        f"where the call ends with the following outcome:\n\n"
        f"  Disposition L1 : {state['l1']}\n"
        f"  Disposition L2 : {state['l2']}\n"
        f"  What this means: {state['description']}\n\n"
        f"The call should feel like a real interaction — invent a plausible customer name, "
        f"loan amount, due date, and account situation. Do not label these as metadata.\n\n"
        f"Once written, also determine:\n"
        f"  call_stage — where in the collections lifecycle this call sits "
        f"(Early Bucket / Mid Bucket / NPA / Legal)\n"
        f"  dialect    — the customer's regional speech pattern "
        f"(e.g. Gujarati-accented Hindi, Tamil-accented English, Punjabi Hinglish)\n"
        f"  l3         — all interaction-level flags from the list below that are directly "
        f"evidenced in the transcript (pipe-separated):\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Formatting rules:\n{rules}\n"
    )


def contacted_prompt_v3(state: dict, rules: str) -> str:
    return (
        f"Task: produce a single debt collection call transcript.\n\n"
        f"Hard constraints:\n"
        f"  Language  : {state['language']} — Roman script only, no Devanagari\n"
        f"  L1 result : {state['l1']}\n"
        f"  L2 result : {state['l2']}\n"
        f"  Scenario  : {state['description']}\n\n"
        f"The dialogue must arrive at this outcome naturally — "
        f"do not announce the disposition, let it emerge from the conversation.\n\n"
        f"Invent realistic context: a common Indian name, a plausible EMI amount and due date, "
        f"and a believable account situation. Weave these into speech, not headers.\n\n"
        f"After writing the transcript, assign three metadata fields:\n"
        f"  call_stage : collections lifecycle stage (Early Bucket / Mid Bucket / NPA / Legal)\n"
        f"  dialect    : customer's spoken variety "
        f"(e.g. Bengali-accented Hindi, Marathi Hinglish, South Indian English)\n"
        f"  l3         : pipe-separated interaction flags evidenced in the transcript, "
        f"chosen from:\n{L3_OPTIONS}\n\n"
        f"Transcript rules:\n{rules}\n"
    )


def contacted_prompt_v4(state: dict, rules: str) -> str:
    return (
        f"Imagine a real phone call between an Indian loan collections agent and a borrower. "
        f"The call is conducted in {state['language']}, written in Roman script (no Devanagari).\n\n"
        f"By the end of the call, the agent logs the interaction as:\n"
        f"  L1 : {state['l1']}\n"
        f"  L2 : {state['l2']}\n\n"
        f"Context: {state['description']}\n\n"
        f"Write the full transcript of this call as it would actually sound — "
        f"natural pauses, overlaps, regional speech patterns, code-switching where realistic. "
        f"Invent the borrower's name, outstanding amount, payment due date, and any relevant "
        f"account background. Do not present these as a list — embed them in the dialogue.\n\n"
        f"Also tag:\n"
        f"  call_stage : the collections stage (Early Bucket / Mid Bucket / NPA / Legal)\n"
        f"  dialect    : the borrower's regional speech variety\n"
        f"  l3         : all applicable flags from the list below, pipe-separated, "
        f"that are directly evidenced in what was said:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Rules to follow:\n{rules}\n"
    )


def contacted_prompt_v5(state: dict, rules: str) -> str:
    return (
        f"You are a data annotator creating synthetic call transcripts for an NLP pipeline.\n\n"
        f"Produce one transcript of a debt collection call in {state['language']} "
        f"using Roman script only — no Devanagari characters anywhere in the output.\n\n"
        f"Classification:\n"
        f"  L1 Disposition : {state['l1']}\n"
        f"  L2 Disposition : {state['l2']}\n"
        f"  Description    : {state['description']}\n\n"
        f"Requirements:\n"
        f"  - The transcript must authentically reflect the L2 outcome through the dialogue itself.\n"
        f"  - Invent a believable Indian borrower profile: name, loan amount, EMI due date, "
        f"and any account details relevant to the scenario. Embed these in the conversation naturally.\n"
        f"  - Assign call_stage (Early Bucket / Mid Bucket / NPA / Legal) based on the "
        f"urgency and tone of the call.\n"
        f"  - Assign dialect to reflect the borrower's regional speech "
        f"(e.g. UP Hindi, Hyderabadi Hinglish, Kerala-accented English).\n"
        f"  - Assign l3 flags (pipe-separated) from the options below — only those "
        f"directly supported by evidence in the transcript:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Transcript formatting rules:\n{rules}\n"
    )


# ── Non-contacted prompt variants ──────────────────────────────────────────────

def non_contacted_prompt_v1(state: dict, rules: str) -> str:
    return (
        f"Generate a realistic debt collection call transcript in {state['language']} using Roman script only (no Devanagari).\n\n"
        f"Outcome to reflect:\n"
        f"- L1          : {state['l1']}\n"
        f"- L2          : {state['l2']}\n"
        f"- Description : {state['description']}\n\n"
        f"Also assign:\n"
        f"- call_stage : the collection stage (e.g. Early Bucket, Mid Bucket, NPA, Legal)\n"
        f"- dialect    : regional Indian dialect of the customer\n"
        f"- l3         : applicable L3 labels (pipe-separated) from:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Rules:\n{rules}\n"
    )


def non_contacted_prompt_v2(state: dict, rules: str) -> str:
    return (
        f"A collections agent places an outbound call in {state['language']} (Roman script only — no Devanagari). "
        f"The call is not answered by the customer.\n\n"
        f"Disposition:\n"
        f"  L1 : {state['l1']}\n"
        f"  L2 : {state['l2']}\n"
        f"  What happened : {state['description']}\n\n"
        f"Write exactly what the agent hears on their handset — a single telecom or voicemail "
        f"system message in {state['language']}. The agent says nothing.\n\n"
        f"Assign:\n"
        f"  call_stage : collections lifecycle stage\n"
        f"  dialect    : regional speech variety\n"
        f"  l3         : applicable flags pipe-separated from:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Rules:\n{rules}\n"
    )


def non_contacted_prompt_v3(state: dict, rules: str) -> str:
    return (
        f"Task: produce a non-contacted call record for a collections dataset.\n\n"
        f"Hard constraints:\n"
        f"  Language     : {state['language']} — Roman script only, no Devanagari\n"
        f"  L1 result    : {state['l1']}\n"
        f"  L2 result    : {state['l2']}\n"
        f"  Scenario     : {state['description']}\n\n"
        f"The agent never speaks. The transcript is one line: a 'Customer:' turn containing "
        f"the automated telecom message the agent would hear — nothing else.\n\n"
        f"Assign call_stage, dialect, and l3 (pipe-separated) from:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Rules:\n{rules}\n"
    )


def non_contacted_prompt_v4(state: dict, rules: str) -> str:
    return (
        f"Imagine a collections agent dials a borrower's number. The call never connects to a human. "
        f"All text must be in {state['language']} using Roman script only — no Devanagari.\n\n"
        f"The outcome is logged as:\n"
        f"  L1 : {state['l1']}\n"
        f"  L2 : {state['l2']}\n\n"
        f"Situation: {state['description']}\n\n"
        f"Write the one-line transcript representing what the agent heard — "
        f"a realistic Indian telecom announcement. "
        f"No agent dialogue. No invented customer conversation.\n\n"
        f"Also tag call_stage, dialect, and l3 from:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Rules:\n{rules}\n"
    )


def non_contacted_prompt_v5(state: dict, rules: str) -> str:
    return (
        f"You are annotating a synthetic call log for a collections NLP model.\n\n"
        f"This record is a non-contact event in {state['language']} (Roman script only — no Devanagari):\n"
        f"  L1 : {state['l1']}\n"
        f"  L2 : {state['l2']}\n"
        f"  Description : {state['description']}\n\n"
        f"The transcript contains exactly one entry — a 'Customer:' turn with the "
        f"automated telecom or voicemail announcement the agent heard. "
        f"There is no agent speech and no human dialogue.\n\n"
        f"Assign call_stage, dialect, and l3 flags (pipe-separated) evidenced in the record:\n\n"
        f"{L3_OPTIONS}\n\n"
        f"Transcript rules:\n{rules}\n"
    )


# ── Variant pools ──────────────────────────────────────────────────────────────
CONTACTED_PROMPT_VARIANTS     = [contacted_prompt_v1, contacted_prompt_v2,
                                  contacted_prompt_v3, contacted_prompt_v4,
                                  contacted_prompt_v5]

NON_CONTACTED_PROMPT_VARIANTS = [non_contacted_prompt_v1, non_contacted_prompt_v2,
                                  non_contacted_prompt_v3, non_contacted_prompt_v4,
                                  non_contacted_prompt_v5]


def build_generation_prompt(state: dict, rules: str, is_contacted: bool) -> str:
    pool = CONTACTED_PROMPT_VARIANTS if is_contacted else NON_CONTACTED_PROMPT_VARIANTS
    variant_fn = random.choice(pool)
    return variant_fn(state, rules)


# ── Nodes ──────────────────────────────────────────────────────────────────────

def generate_transcript(state: TranscriptState):
    is_contacted = state["l1"].strip().lower() == "contacted"
    rules = (
        CONTACTED_RULES.format(language=state["language"])
        if is_contacted
        else NON_CONTACTED_RULES.format(language=state["language"])
    )

    human_content = build_generation_prompt(state, rules, is_contacted)

    messages = [
        SystemMessage(content=(
            "You are an expert dataset generator for loan collection call transcripts.\n\n"
            "CRITICAL SCRIPT RULE: All output must use Roman script (A-Z letters) only. "
            "Never use Devanagari or any other non-Latin script, even when writing Hindi or Hinglish. "
            "Hindi and Hinglish must be fully transliterated into Roman characters.\n\n"
            f"Full Disposition Taxonomy for reference:\n{DISPOSITION_TAXONOMY}\n"
        )),
        HumanMessage(content=human_content),
    ]
    response = structured_generator_llm.invoke(messages)
    return {
        "transcript":         response.transcript,
        "call_stage":         response.call_stage,
        "dialect":            response.dialect,
        "l3":                 response.l3,
        "transcript_history": [response.transcript],
    }


def evaluate_transcript(state: TranscriptState):
    is_contacted = state["l1"].strip().lower() == "contacted"

    if is_contacted:
        checklist = (
            "1. DIARISATION      - Every turn is labelled strictly as 'Agent:' or 'Customer:'. No missing or inconsistent labels.\n"
            f"2. LANGUAGE         - Entire transcript is written in {state['language']} with no unjustified language switches.\n"
            "3. ROMAN SCRIPT     - No Devanagari or non-Latin characters anywhere in the transcript.\n"
            f"4. OUTCOME          - The conversation outcome matches L1 \"{state['l1']}\" and L2 \"{state['l2']}\" as described: \"{state['description']}\".\n"
            f"5. L3 LABELS        - Each assigned L3 label ({state['l3']}) is directly evidenced by something said or done in the transcript.\n"
            f"6. CALL STAGE       - The tone, urgency, and context of the call matches the assigned call stage \"{state['call_stage']}\".\n"
            f"7. DIALECT          - The customer's language style reflects the assigned dialect \"{state['dialect']}\".\n"
            "8. LENGTH           - Transcript is between 1000 and 4500 characters.\n"
            "9. NOISE & REALISM  - Includes at least one of: interruption, incomplete sentence, repetition, or natural code-switching.\n"
            "10. INVENTED DETAILS - Customer name, loan amount, due date appear naturally in dialogue, not as metadata headers.\n"
        )
    else:
        checklist = (
            "1. NO AGENT SPEECH       - There must be zero 'Agent:' turns. The agent dials but never speaks.\n"
            "2. SINGLE CUSTOMER TURN  - The transcript contains exactly one 'Customer:' turn with a telecom/voicemail announcement only.\n"
            f"3. ANNOUNCEMENT REALISM  - The announcement is a realistic Indian telecom message matching L2 \"{state['l2']}\" written in {state['language']}.\n"
            "4. NO INVENTED DIALOGUE  - No loan details, no back-and-forth conversation. Only the system announcement.\n"
            "5. ROMAN SCRIPT          - No Devanagari or non-Latin characters anywhere in the transcript.\n"
        )

    messages = [
        SystemMessage(content=(
            "You are a strict QA auditor for synthetic debt collection call datasets in the Indian market.\n"
            "You are NOT generating or rewriting — you are auditing an already generated transcript against a fixed checklist.\n"
            "Your job is to find failures, not to suggest rewrites."
        )),
        HumanMessage(content=(
            f"Audit the following generated transcript against each checkpoint below.\n\n"
            f"--- METADATA ---\n"
            f"Language    : {state['language']}\n"
            f"Call Stage  : {state['call_stage']}\n"
            f"Dialect     : {state['dialect']}\n"
            f"L1          : {state['l1']}\n"
            f"L2          : {state['l2']}\n"
            f"Description : {state['description']}\n"
            f"L3 Assigned : {state['l3']}\n\n"
            f"--- TRANSCRIPT ---\n"
            f"{state['transcript']}\n\n"
            f"--- AUDIT CHECKLIST ---\n"
            f"Go through each point and mark PASS or FAIL:\n\n"
            f"{checklist}\n"
            f"--- DECISION ---\n"
            f"If ALL checkpoints PASS -> evaluation: \"approved\"\n"
            f"If ANY checkpoint FAILS  -> evaluation: \"needs_improvement\" and state exactly which checkpoints failed and why.\n\n"
            f"Respond ONLY in structured format:\n"
            f"- evaluation: \"approved\" or \"needs_improvement\"\n"
            f"- feedback: List each failed checkpoint with a one-line reason. If approved, write \"All checkpoints passed.\"\n"
        )),
    ]
    response = structured_evaluator_llm.invoke(messages)
    return {
        "evaluation":       response.evaluation,
        "feedback":         response.feedback,
        "feedback_history": [response.feedback],
    }


def optimize_transcript(state: TranscriptState):
    is_contacted = state["l1"].strip().lower() == "contacted"
    rules = (
        CONTACTED_RULES.format(language=state["language"])
        if is_contacted
        else NON_CONTACTED_RULES.format(language=state["language"])
    )

    fresh_prompt = build_generation_prompt(state, rules, is_contacted)

    messages = [
        SystemMessage(content=(
            "You rewrite debt collection call transcripts based on QA audit feedback to fix specific failures.\n\n"
            "CRITICAL SCRIPT RULE: All output must use Roman script (A-Z letters) only. "
            "Never use Devanagari or any other non-Latin script, even when writing Hindi or Hinglish."
        )),
        HumanMessage(content=(
            f"The following transcript failed QA. Rewrite it to fix every issue listed in the feedback.\n\n"
            f"--- FEEDBACK ---\n{state['feedback']}\n\n"
            f"--- METADATA TO PRESERVE ---\n"
            f"Language    : {state['language']}\n"
            f"Call Stage  : {state['call_stage']}\n"
            f"Dialect     : {state['dialect']}\n"
            f"L1          : {state['l1']}\n"
            f"L2          : {state['l2']}\n"
            f"Description : {state['description']}\n\n"
            f"--- ORIGINAL TRANSCRIPT ---\n{state['transcript']}\n\n"
            f"--- REWRITE INSTRUCTIONS ---\n{fresh_prompt}\n"
        )),
    ]
    response = structured_generator_llm.invoke(messages)
    return {
        "transcript":         response.transcript,
        "call_stage":         response.call_stage,
        "dialect":            response.dialect,
        "l3":                 response.l3,
        "iteration":          state["iteration"] + 1,
        "transcript_history": [response.transcript],
    }


def route_evaluation(state: TranscriptState):
    if state["iteration"] >= state["max_iteration"]:
        return "approved"
    if state["evaluation"] == "approved":
        return "approved"
    return "needs_improvement"


# ── Graph ──────────────────────────────────────────────────────────────────────
graph = StateGraph(TranscriptState)
graph.add_node("generate", generate_transcript)
graph.add_node("evaluate", evaluate_transcript)
graph.add_node("optimize", optimize_transcript)
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate",
    route_evaluation,
    {"approved": END, "needs_improvement": "optimize"}
)
graph.add_edge("optimize", "evaluate")
workflow = graph.compile()

# ── Batch helpers ──────────────────────────────────────────────────────────────

def build_disposition_list() -> list[dict]:
    rows = df[["Level 1", "Level 2", "Description"]].to_dict("records")
    base, remainder = divmod(BATCH_SIZE, len(rows))
    assignments = []
    for i, row in enumerate(rows):
        count = base + (1 if i < remainder else 0)
        assignments.extend([{
            "l1":          row["Level 1"],
            "l2":          row["Level 2"],
            "description": row["Description"],
        }] * count)
    random.shuffle(assignments)
    return assignments


def run_batch() -> dict:
    hindi_count    = int(BATCH_SIZE * 0.40)
    hinglish_count = int(BATCH_SIZE * 0.35)
    english_count  = BATCH_SIZE - hindi_count - hinglish_count

    languages = (
        ["Hindi"]    * hindi_count +
        ["Hinglish"] * hinglish_count +
        ["English"]  * english_count
    )
    random.shuffle(languages)
    disposition_list = build_disposition_list()

    print(f"\nBatch: {BATCH_SIZE} | Hindi: {hindi_count} | Hinglish: {hinglish_count} | English: {english_count}")
    print(f"Max concurrency: {MAX_CONCURRENCY} | Max iterations: {MAX_ITERATION}\n")

    inputs = [
        {
            "language":      lang,
            "l1":            disp["l1"],
            "l2":            disp["l2"],
            "description":   disp["description"],
            "l3":            "",
            "call_stage":    "",
            "dialect":       "",
            "transcript":    "",
            "evaluation":    "needs_improvement",
            "feedback":      "",
            "iteration":     1,
            "max_iteration": MAX_ITERATION,
        }
        for lang, disp in zip(languages, disposition_list)
    ]

    start_time   = time.time()
    final_states = workflow.batch(inputs, config={"max_concurrency": MAX_CONCURRENCY})
    elapsed      = time.time() - start_time

    results, approved, needs_improvement = [], [], []

    for i, (result, lang, disp) in enumerate(zip(final_states, languages, disposition_list)):
        entry = {
            "id":          i + 1,
            "call_stage":  result.get("call_stage", ""),
            "language":    lang,
            "dialect":     result.get("dialect", ""),
            "transcript":  result["transcript"],
            "L1":          disp["l1"],
            "L2":          disp["l2"],
            "L3":          result.get("l3", ""),
            "Description": disp["description"],
            "evaluation":  result["evaluation"],
        }
        results.append(entry)
        (approved if result["evaluation"] == "approved" else needs_improvement).append(entry)
        print(
            f"[{i+1}/{BATCH_SIZE}] {result['evaluation'].upper():<20} "
            f"| {lang:<10} | {disp['l1']} / {disp['l2']}"
        )

    print(f"\nDone in {elapsed:.1f}s | Approved: {len(approved)} | Needs improvement: {len(needs_improvement)}")
    return {"all": results, "approved": approved, "needs_improvement": needs_improvement}


# ── Format results ─────────────────────────────────────────────────────────────

def format_results(batch_results: dict) -> list[dict]:
    return [
        {
            "id":         f"call_{entry['id']:03d}",
            "language":   entry["language"],
            "dialect":    entry["dialect"],
            "call_stage": entry["call_stage"],
            "transcript": entry["transcript"].strip(),
            "L1":         entry["L1"],
            "L2":         entry["L2"],
            "L3":         entry["L3"],
            "evaluation": entry["evaluation"],
        }
        for entry in batch_results["all"]
    ]


# ── Run ────────────────────────────────────────────────────────────────────────
batch_results = run_batch()

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path   = f"transcripts_{timestamp}.json"
output_data = format_results(batch_results)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(output_data)} transcripts → {json_path}")


Batch: 150 | Hindi: 60 | Hinglish: 52 | English: 38
Max concurrency: 25 | Max iterations: 2



BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaMtrXwFKgJ5MRrYvnM5V'}